# H&M Fashion Recommendation System
## End-to-End Implementation with PySpark & Apple Silicon Optimization

This notebook runs the complete recommendation system pipeline:
1. **Stage 1**: Data Loading with PySpark (50% sample)
2. **Stage 2**: Candidate Generation with PySpark
3. **Stage 3**: Feature Engineering with PySpark
4. **Stage 4A**: LightGBM Model Training
5. **Stage 4B**: Two-Tower Neural Network (MPS accelerated)
6. **Stage 7**: Evaluation with MAP@12

**Optimized for Apple Silicon (M1/M2/M3/M4)** with MPS acceleration for PyTorch.


## 0. Environment Setup & Verification


In [ ]:
# System information
import platform
import sys
import os

print("=" * 60)
print("SYSTEM INFORMATION")
print("=" * 60)
print(f"Python: {sys.version}")
print(f"Platform: {platform.platform()}")
print(f"Processor: {platform.processor()}")
print(f"Machine: {platform.machine()}")


In [ ]:
# Verify Apple Silicon MPS availability
import torch

print("\n" + "=" * 60)
print("PYTORCH & MPS STATUS")
print("=" * 60)
print(f"PyTorch version: {torch.__version__}")
print(f"MPS (Metal) available: {torch.backends.mps.is_available()}")
print(f"MPS built: {torch.backends.mps.is_built()}")

if torch.backends.mps.is_available():
    device = torch.device('mps')
    print(f"\nUsing Apple Silicon MPS acceleration!")
    # Test MPS
    x = torch.randn(100, 100, device=device)
    y = x @ x.T
    print(f"   MPS test passed: created {x.shape} tensor")
else:
    device = torch.device('cpu')
    print(f"\nMPS not available, using CPU")


In [ ]:
# Verify PySpark availability
try:
    from pyspark.sql import SparkSession
    print("\n" + "=" * 60)
    print("PYSPARK STATUS")
    print("=" * 60)
    print(f"PySpark imported successfully")
    
    # Quick Spark test
    spark_test = SparkSession.builder.appName("test").getOrCreate()
    print(f"   Spark version: {spark_test.version}")
    spark_test.stop()
    print(f"   Spark test passed!")
except ImportError as e:
    print(f"PySpark not available: {e}")
    print("Install with: pip install pyspark")


In [ ]:
# Core imports
import pandas as pd
import numpy as np
import lightgbm as lgb
from datetime import datetime, timedelta
from pathlib import Path
import warnings
import gc
import time

warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)

print("\n" + "=" * 60)
print("IMPORTS COMPLETE")
print("=" * 60)
print(f"Pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")
print(f"LightGBM: {lgb.__version__}")


In [ ]:
# Change to project directory
import os
os.chdir('/Users/raghu/coding/fashion_recommendation_system')
print(f"Working directory: {os.getcwd()}")

# Import project modules
from config import Config, config, LightGBMConfig, NeuralTowerConfig, EvaluationConfig
from utils import print_section, force_garbage_collection, print_memory
from metrics import (
    calculate_map_at_k, calculate_precision_at_k,
    calculate_recall_at_k, calculate_ndcg_at_k,
    evaluate_all_metrics, evaluate_map_at_12
)

print(f"\nConfiguration loaded:")
print(f"  DATA_PATH: {config.DATA_PATH}")
print(f"  OUTPUT_PATH: {config.OUTPUT_PATH}")
print(f"  SAMPLE_FRACTION: {config.SAMPLE_FRACTION} (50% of data)")
print(f"  SPARK_MEMORY: {config.SPARK_MEMORY}")


---
## 1. Stage 1: Data Loading with PySpark

Load the H&M dataset using PySpark for distributed processing.
- Uses **50% stratified sampling** instead of fixed user count
- Temporal windowing for train/val split
- Memory-optimized output


In [ ]:
# Track total time
pipeline_start = time.time()

print("=" * 80)
print("STAGE 1: DATA LOADING WITH PYSPARK")
print("=" * 80)

stage1_start = time.time()

from stage1_load_data import run_stage1

# Run Stage 1
data = run_stage1()

stage1_time = time.time() - stage1_start
print(f"\nStage 1 completed in {stage1_time:.1f} seconds ({stage1_time/60:.1f} minutes)")


In [ ]:
# Display Stage 1 Results
print("\n" + "=" * 60)
print("STAGE 1 RESULTS SUMMARY")
print("=" * 60)

print(f"\nDatasets loaded:")
print(f"  Train transactions: {len(data['train_transactions']):,} rows")
print(f"  Val transactions: {len(data['val_transactions']):,} rows")
print(f"  Customers: {len(data['customers']):,} rows")
print(f"  Articles: {len(data['articles']):,} rows")

print(f"\nUser/Item counts:")
print(f"  Unique users: {len(data['all_users']):,}")
print(f"  Unique items: {len(data['all_items']):,}")
print(f"  Validation users: {len(data['val_users']):,}")

print(f"\nDate range:")
print(f"  Max date: {data['max_date']}")

# Memory usage
print(f"\nMemory usage:")
for name, df in [('train_transactions', data['train_transactions']), 
                  ('val_transactions', data['val_transactions']),
                  ('customers', data['customers']),
                  ('articles', data['articles'])]:
    mem_mb = df.memory_usage(deep=True).sum() / 1024**2
    print(f"  {name}: {mem_mb:.1f} MB")


---
## 2. Stage 2: Candidate Generation with PySpark

Generate candidates using 5 strategies:
1. **Repurchase**: Items user bought before
2. **Popularity**: Trending items
3. **Co-purchase**: Item-to-item CF
4. **User-KNN**: User-based CF
5. **Category**: Category preferences


In [ ]:
print("\n" + "=" * 80)
print("STAGE 2: CANDIDATE GENERATION WITH PYSPARK")
print("=" * 80)

stage2_start = time.time()

from stage2_candidates import run_stage2

# Run Stage 2
candidates = run_stage2(data)

stage2_time = time.time() - stage2_start
print(f"\nStage 2 completed in {stage2_time:.1f} seconds ({stage2_time/60:.1f} minutes)")


In [ ]:
# Display Stage 2 Results
print("\n" + "=" * 60)
print("STAGE 2 RESULTS SUMMARY")
print("=" * 60)

print(f"\nCandidate statistics:")
print(f"  Total candidates: {len(candidates):,}")
print(f"  Unique users: {candidates['customer_id'].nunique():,}")
print(f"  Unique items: {candidates['article_id'].nunique():,}")
print(f"  Candidates per user: {len(candidates) / candidates['customer_id'].nunique():.1f}")

# Strategy coverage
print(f"\nStrategy coverage:")
score_cols = ['repurchase_score', 'popularity_score', 'copurchase_score',
              'userknn_score', 'category_score']
for col in score_cols:
    if col in candidates.columns:
        coverage = (candidates[col] > 0).mean() * 100
        print(f"  {col}: {coverage:.1f}%")

# Number of strategies distribution
if 'n_strategies' in candidates.columns:
    print(f"\nStrategies per candidate:")
    print(candidates['n_strategies'].value_counts().sort_index())


---
## 3. Stage 3: Feature Engineering with PySpark

Extract features using PySpark:
- **User features**: Purchase history, demographics
- **Item features**: Sales stats, category metadata
- **Interaction features**: User-item relationships


In [ ]:
print("\n" + "=" * 80)
print("STAGE 3: FEATURE ENGINEERING WITH PYSPARK")
print("=" * 80)

stage3_start = time.time()

from stage3_features import run_stage3

# Add candidates to data dict
data['candidates'] = candidates

# Run Stage 3
train_data, val_data = run_stage3(data)

stage3_time = time.time() - stage3_start
print(f"\nStage 3 completed in {stage3_time:.1f} seconds ({stage3_time/60:.1f} minutes)")


In [ ]:
# Display Stage 3 Results
print("\n" + "=" * 60)
print("STAGE 3 RESULTS SUMMARY")
print("=" * 60)

# Count features
exclude_cols = ['customer_id', 'article_id', 'label', 'user_type']
feature_cols = [c for c in train_data.columns if c not in exclude_cols]

print(f"\nDataset sizes:")
print(f"  Training samples: {len(train_data):,}")
print(f"  Validation samples: {len(val_data):,}")
print(f"  Total features: {len(feature_cols)}")

print(f"\nLabel distribution:")
print(f"  Train positives: {train_data['label'].sum():,} ({100*train_data['label'].mean():.2f}%)")
print(f"  Val positives: {val_data['label'].sum():,} ({100*val_data['label'].mean():.2f}%)")

print(f"\nUser counts:")
print(f"  Training users: {train_data['customer_id'].nunique():,}")
print(f"  Validation users: {val_data['customer_id'].nunique():,}")

# Sample features
print(f"\nSample features (first 10):")
for i, col in enumerate(feature_cols[:10]):
    print(f"  {i+1}. {col}")


In [ ]:
# Clear memory before model training
del data, candidates
force_garbage_collection()
print_memory()


---
## 4A. Stage 4A: LightGBM Model Training

Train gradient boosting models:
- Binary classifier
- LambdaRank ranker
- XENDCG ranker
- Deep classifier


In [ ]:
print("\n" + "=" * 80)
print("STAGE 4A: LIGHTGBM MODEL TRAINING")
print("=" * 80)

stage4a_start = time.time()

from stage4a_lightgbm import run_stage4a

# Run Stage 4A
lgb_results = run_stage4a()

stage4a_time = time.time() - stage4a_start
print(f"\nStage 4A completed in {stage4a_time:.1f} seconds ({stage4a_time/60:.1f} minutes)")


In [ ]:
# Display LightGBM Results
print("\n" + "=" * 60)
print("LIGHTGBM RESULTS SUMMARY")
print("=" * 60)

if lgb_results and 'model_scores' in lgb_results:
    print("\nModel MAP@12 Scores:")
    for model_name, score in sorted(lgb_results['model_scores'].items(), 
                                    key=lambda x: x[1], reverse=True):
        print(f"  {model_name}: {score:.6f}")
    
    if 'ensemble_score' in lgb_results:
        print(f"\n  Ensemble: {lgb_results['ensemble_score']:.6f}")

# Feature importance
if lgb_results and 'feature_importance' in lgb_results:
    print("\nTop 10 Important Features:")
    fi = lgb_results['feature_importance']
    for i, (feat, imp) in enumerate(fi[:10], 1):
        print(f"  {i}. {feat}: {imp:.0f}")


---
## 4B. Stage 4B: Two-Tower Neural Network (MPS Accelerated)

Train deep learning model with Apple Silicon optimization:
- **User Tower**: User features → User embedding
- **Item Tower**: Item features → Item embedding  
- **Fusion Layer**: Combined prediction

**Using MPS (Metal Performance Shaders) for acceleration**


In [ ]:
print("\n" + "=" * 80)
print("STAGE 4B: TWO-TOWER NEURAL NETWORK (MPS)")
print("=" * 80)

# Verify MPS is available
import torch
if torch.backends.mps.is_available():
    print("Using Apple Silicon MPS acceleration")
else:
    print("MPS not available, using CPU")

stage4b_start = time.time()

from stage4b_neural import run_stage4b

# Run Stage 4B
neural_results = run_stage4b()

stage4b_time = time.time() - stage4b_start
print(f"\nStage 4B completed in {stage4b_time:.1f} seconds ({stage4b_time/60:.1f} minutes)")


In [ ]:
# Display Neural Network Results
print("\n" + "=" * 60)
print("NEURAL NETWORK RESULTS SUMMARY")
print("=" * 60)

if neural_results:
    if 'best_map12' in neural_results:
        print(f"\nBest Validation MAP@12: {neural_results['best_map12']:.6f}")
    
    if 'training_history' in neural_results:
        history = neural_results['training_history']
        print(f"\nTraining history:")
        print(f"  Epochs trained: {len(history.get('train_loss', []))}")
        if 'train_loss' in history:
            print(f"  Final train loss: {history['train_loss'][-1]:.4f}")
        if 'val_map12' in history:
            print(f"  Final val MAP@12: {history['val_map12'][-1]:.6f}")


---
## 5. Stage 7: Evaluation with MAP@12

Final evaluation and comparison:
- Evaluate all models with MAP@12, Precision@K, Recall@K, NDCG@K
- Create final ensemble
- Generate submission file


In [ ]:
print("\n" + "=" * 80)
print("STAGE 7: EVALUATION WITH MAP@12")
print("=" * 80)

stage7_start = time.time()

from stage7_evaluation import run_stage7

# Run Stage 7
eval_results = run_stage7()

stage7_time = time.time() - stage7_start
print(f"\nStage 7 completed in {stage7_time:.1f} seconds ({stage7_time/60:.1f} minutes)")


In [ ]:
# Display Final Evaluation Results
print("\n" + "=" * 80)
print("FINAL EVALUATION RESULTS")
print("=" * 80)

if eval_results and 'comparison_df' in eval_results:
    comparison_df = eval_results['comparison_df']
    
    print("\n" + "-" * 80)
    print("MODEL COMPARISON (sorted by MAP@12)")
    print("-" * 80)
    
    # Display key metrics
    display_cols = ['MAP@12', 'Precision@12', 'Recall@12', 'NDCG@12']
    available_cols = [c for c in display_cols if c in comparison_df.columns]
    
    if available_cols:
        print(comparison_df[available_cols].to_string())
    else:
        print(comparison_df.to_string())
    
    # Best model
    if 'MAP@12' in comparison_df.columns:
        best_model = comparison_df['MAP@12'].idxmax()
        best_score = comparison_df.loc[best_model, 'MAP@12']
        print(f"\n🏆 Best Model: {best_model}")
        print(f"   MAP@12: {best_score:.6f}")


In [ ]:
# Display metrics at different K values
print("\n" + "=" * 60)
print("METRICS AT DIFFERENT K VALUES")
print("=" * 60)

if eval_results and 'evaluation_results' in eval_results:
    eval_dict = eval_results['evaluation_results']
    
    # Get the best model's results or final ensemble
    if 'final_ensemble' in eval_dict:
        results = eval_dict['final_ensemble']
        model_name = 'Final Ensemble'
    else:
        model_name = list(eval_dict.keys())[0]
        results = eval_dict[model_name]
    
    print(f"\n{model_name} Performance:")
    print("-" * 50)
    
    k_values = [1, 3, 5, 10, 12]
    print(f"{'K':<5} {'MAP@K':<12} {'P@K':<12} {'R@K':<12} {'NDCG@K':<12}")
    print("-" * 50)
    
    for k in k_values:
        map_k = results.get(f'MAP@{k}', 0)
        p_k = results.get(f'Precision@{k}', 0)
        r_k = results.get(f'Recall@{k}', 0)
        ndcg_k = results.get(f'NDCG@{k}', 0)
        print(f"{k:<5} {map_k:<12.6f} {p_k:<12.6f} {r_k:<12.6f} {ndcg_k:<12.6f}")


---
## 6. Pipeline Summary


In [ ]:
# Calculate total time
total_time = time.time() - pipeline_start

print("\n" + "=" * 80)
print("PIPELINE EXECUTION COMPLETE")
print("=" * 80)

print("\nStage Timing Summary:")
print("-" * 60)
print(f"  Stage 1 (Data Loading):      {stage1_time:>8.1f}s ({stage1_time/60:>5.1f} min)")
print(f"  Stage 2 (Candidates):        {stage2_time:>8.1f}s ({stage2_time/60:>5.1f} min)")
print(f"  Stage 3 (Features):          {stage3_time:>8.1f}s ({stage3_time/60:>5.1f} min)")
print(f"  Stage 4A (LightGBM):         {stage4a_time:>8.1f}s ({stage4a_time/60:>5.1f} min)")
print(f"  Stage 4B (Neural):           {stage4b_time:>8.1f}s ({stage4b_time/60:>5.1f} min)")
print(f"  Stage 7 (Evaluation):        {stage7_time:>8.1f}s ({stage7_time/60:>5.1f} min)")
print("-" * 60)
print(f"  TOTAL:                       {total_time:>8.1f}s ({total_time/60:>5.1f} min)")

# PySpark vs Pandas processing
pyspark_time = stage1_time + stage2_time + stage3_time
ml_time = stage4a_time + stage4b_time

print(f"\nProcessing Breakdown:")
print(f"  PySpark stages (1-3):  {pyspark_time:>8.1f}s ({100*pyspark_time/total_time:>5.1f}%)")
print(f"  ML training (4A-4B):   {ml_time:>8.1f}s ({100*ml_time/total_time:>5.1f}%)")
print(f"  Evaluation (7):        {stage7_time:>8.1f}s ({100*stage7_time/total_time:>5.1f}%)")


In [ ]:
# Final summary - Output files
print("\n" + "=" * 80)
print("OUTPUT FILES GENERATED")
print("=" * 80)

output_files = [
    ('train_transactions.parquet', config.OUTPUT_PATH),
    ('val_transactions.parquet', config.OUTPUT_PATH),
    ('candidates.parquet', config.OUTPUT_PATH),
    ('training_features.parquet', config.OUTPUT_PATH),
    ('train_data.parquet', config.MODEL_PATH),
    ('val_data.parquet', config.MODEL_PATH),
    ('model_comparison.csv', config.MODEL_PATH),
    ('submission.csv', config.MODEL_PATH),
]

print(f"\nOutput directory: {config.OUTPUT_PATH}")
print(f"Model directory: {config.MODEL_PATH}")
print("\nGenerated files:")

for filename, directory in output_files:
    filepath = directory / filename
    if filepath.exists():
        size_mb = filepath.stat().st_size / 1024**2
        print(f"{filename}: {size_mb:.1f} MB")
    else:
        print(f"{filename}: not found")


In [ ]:
# Display final MAP@12 results with rankings
print("\n" + "=" * 80)
print("FINAL MAP@12 RESULTS")
print("=" * 80)

if eval_results and 'comparison_df' in eval_results:
    comparison_df = eval_results['comparison_df']
    
    print("\nModel Rankings:")
    print("-" * 50)
    
    for rank, (model_name, row) in enumerate(comparison_df.iterrows(), 1):
        medal = "🥇" if rank == 1 else "🥈" if rank == 2 else "🥉" if rank == 3 else f"{rank}."
        map12 = row.get('MAP@12', 0)
        print(f"  {medal} {model_name}: MAP@12 = {map12:.6f}")
    
    # Best result highlight
    if 'MAP@12' in comparison_df.columns:
        best_score = comparison_df['MAP@12'].max()
        print(f"\n" + "*" * 50)
        print(f"  ⭐ BEST MAP@12: {best_score:.6f}")
        print("*" * 50)


---
## 7. Visualizations


In [ ]:
import matplotlib.pyplot as plt

# Create visualization of results
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 1. Stage timing
ax1 = axes[0]
stages = ['Stage 1\n(Data)', 'Stage 2\n(Candidates)', 'Stage 3\n(Features)', 
          'Stage 4A\n(LightGBM)', 'Stage 4B\n(Neural)', 'Stage 7\n(Eval)']
times = [stage1_time/60, stage2_time/60, stage3_time/60, 
         stage4a_time/60, stage4b_time/60, stage7_time/60]
colors = ['#3498db', '#3498db', '#3498db', '#e74c3c', '#e74c3c', '#2ecc71']

bars = ax1.bar(stages, times, color=colors, edgecolor='white', linewidth=1.5)
ax1.set_ylabel('Time (minutes)', fontsize=12)
ax1.set_title('Pipeline Stage Timing', fontsize=14, fontweight='bold')
ax1.set_ylim(0, max(times) * 1.2 if times else 1)

for bar, t in zip(bars, times):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, 
             f'{t:.1f}m', ha='center', va='bottom', fontsize=10)

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#3498db', label='PySpark'),
    Patch(facecolor='#e74c3c', label='ML Training'),
    Patch(facecolor='#2ecc71', label='Evaluation')
]
ax1.legend(handles=legend_elements, loc='upper right')

# 2. Model comparison (if available)
ax2 = axes[1]
if eval_results and 'comparison_df' in eval_results:
    comparison_df = eval_results['comparison_df']
    if 'MAP@12' in comparison_df.columns:
        models = comparison_df.index.tolist()
        map12_scores = comparison_df['MAP@12'].tolist()
        
        colors_model = plt.cm.viridis(np.linspace(0.3, 0.9, len(models)))
        bars2 = ax2.barh(models, map12_scores, color=colors_model)
        ax2.set_xlabel('MAP@12', fontsize=12)
        ax2.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
        
        for bar, score in zip(bars2, map12_scores):
            ax2.text(score + 0.001, bar.get_y() + bar.get_height()/2,
                     f'{score:.4f}', va='center', fontsize=10)
else:
    ax2.text(0.5, 0.5, 'No model results available', ha='center', va='center',
             fontsize=14, transform=ax2.transAxes)
    ax2.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')

# 3. Metrics at different K
ax3 = axes[2]
if eval_results and 'evaluation_results' in eval_results:
    eval_dict = eval_results['evaluation_results']
    
    # Get best model or ensemble
    if 'final_ensemble' in eval_dict:
        results = eval_dict['final_ensemble']
    elif len(eval_dict) > 0:
        results = list(eval_dict.values())[0]
    else:
        results = {}
    
    k_values = [1, 3, 5, 10, 12]
    map_scores = [results.get(f'MAP@{k}', 0) for k in k_values]
    precision_scores = [results.get(f'Precision@{k}', 0) for k in k_values]
    recall_scores = [results.get(f'Recall@{k}', 0) for k in k_values]
    
    ax3.plot(k_values, map_scores, 'o-', label='MAP@K', linewidth=2, markersize=8)
    ax3.plot(k_values, precision_scores, 's-', label='Precision@K', linewidth=2, markersize=8)
    ax3.plot(k_values, recall_scores, '^-', label='Recall@K', linewidth=2, markersize=8)
    
    ax3.set_xlabel('K', fontsize=12)
    ax3.set_ylabel('Score', fontsize=12)
    ax3.set_title('Metrics at Different K', fontsize=14, fontweight='bold')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    ax3.set_xticks(k_values)
else:
    ax3.text(0.5, 0.5, 'No evaluation results', ha='center', va='center',
             fontsize=14, transform=ax3.transAxes)

plt.tight_layout()
plt.savefig(config.OUTPUT_PATH / 'pipeline_results.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✅ Saved visualization to {config.OUTPUT_PATH / 'pipeline_results.png'}")


In [ ]:
# Display sample submission
print("\n" + "=" * 60)
print("SAMPLE SUBMISSION PREVIEW")
print("=" * 60)

submission_path = config.MODEL_PATH / 'submission.csv'
if submission_path.exists():
    submission = pd.read_csv(submission_path)
    print(f"\nTotal users in submission: {len(submission):,}")
    print(f"\nSample (first 5 users):")
    print("-" * 60)
    
    for idx, row in submission.head(5).iterrows():
        customer_id = row['customer_id']
        predictions = row['prediction'].split()
        print(f"\nUser: {customer_id[:20]}...")
        print(f"  Top 12 recommendations: {', '.join(predictions[:6])}...")
else:
    print("Submission file not found")


In [ ]:
# Final cleanup
force_garbage_collection()
print("\n Pipeline execution complete!")
print_memory()
